# LLR Comparison: Base ESM2 vs LoRA-Finetuned (per-pfam)

Loads the LoRA delta weights produced by `ESM_domainome_ft_pfam.ipynb` for each pfam,
recomputes per-domain LLR matrices with both the un-finetuned base model and the
finetuned model for that pfam, and uploads one gzipped pickle per pfam to GCS:
`{domain_id: {sequence, seq_offset, base_llr, ft_llr, delta_llr, ...}}`.

Output paths mirror the training run, i.e. one
`llr_comparison.pkl.gz` per pfam under
`gs://{bucket}/{prefix}/global_finetune/{pfam_id}/`.


In [1]:
import os
import io
import gzip
import json
import pickle
import warnings
import logging

import numpy as np
import pandas as pd
import torch

from dotenv import load_dotenv
from tqdm import tqdm

from transformers import EsmForMaskedLM, EsmTokenizer
from peft import LoraConfig, get_peft_model
from google.cloud import storage

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Constants (must match training run) ──────────────────────────────────────

AA_ORDER         = ['L', 'A', 'G', 'V', 'S', 'E', 'R', 'T', 'I', 'D',
                    'P', 'K', 'Q', 'N', 'F', 'Y', 'M', 'H', 'W', 'C']
FITNESS_AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
MODEL_NAME       = "facebook/esm2_t33_650M_UR50D"
MAX_LEN          = 1022
LORA_DROPOUT     = 0.1

# ── Local + GCS layout (must match training run) ─────────────────────────────

# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "plm-study-484223-e1c13b49d132.json"

OUTPUT_DIR     = "finetuned_models_v2"
RUN_ID_PREFIX  = "global_finetune"   # matches training notebook's run_id = f"{RUN_ID_PREFIX}/{pfam_id}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GCS_PROJECT           = "plm-study-484223"
GCS_BUCKET            = "domainome-data"
GCS_DOMAIN_DATA_BLOB  = "ESM2/dict_fitness.pkl.gz"
GCS_FULL_PROTEIN_BLOB = "ESM2/dict_domainome_uniprot_new.pkl.gz"
GCS_OUTPUT_PREFIX     = "ESM2/ft_per_pfam/"   # must end with /

LLR_OUTPUT_NAME = "llr_comparison.pkl.gz"


def pfam_paths(pfam_id):
    """Resolve all per-pfam local + GCS paths from a pfam id."""
    run_id  = f"{RUN_ID_PREFIX}/{pfam_id}"
    run_dir = os.path.join(OUTPUT_DIR, run_id)
    os.makedirs(run_dir, exist_ok=True)
    return {
        "run_id":        run_id,
        "run_dir":       run_dir,
        "metadata_local": os.path.join(run_dir, "metadata.json"),
        "lora_local":    os.path.join(run_dir, "best_model_lora.pth"),
        "llr_local":     os.path.join(run_dir, LLR_OUTPUT_NAME),
        "gcs_metadata":  f"{GCS_OUTPUT_PREFIX}{run_id}/metadata.json",
        "gcs_lora":      f"{GCS_OUTPUT_PREFIX}{run_id}/best_model_lora.pth",
        "gcs_llr":       f"{GCS_OUTPUT_PREFIX}{run_id}/{LLR_OUTPUT_NAME}",
    }


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


In [3]:
# ── GCS Helpers ──────────────────────────────────────────────────────────────
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# _gcs_client = storage.Client(project=GCS_PROJECT) if GCS_PROJECT else storage.Client()
_gcs_client = storage.Client()
_gcs_bucket = _gcs_client.bucket(GCS_BUCKET)


def gcs_download_pickle_gz(blob):
    data = blob.download_as_bytes()
    with gzip.GzipFile(fileobj=io.BytesIO(data), mode="rb") as gz:
        return pickle.load(gz)


def gcs_download_to_file(blob_path, local_path):
    blob = _gcs_bucket.blob(blob_path)
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    blob.download_to_filename(local_path)
    return local_path


def gcs_upload_file(blob_path, local_path, content_type="application/octet-stream"):
    blob = _gcs_bucket.blob(blob_path)
    blob.upload_from_filename(local_path, content_type=content_type)

In [4]:
# ── Helpers (mirrored from training notebook) ────────────────────────────────

def truncate_to_domain(sequence, domain_start, domain_end, max_len=MAX_LEN):
    seq_len    = len(sequence)
    domain_mid = (domain_start + domain_end) // 2
    half       = max_len // 2
    start      = max(0, domain_mid - 1 - half)
    end        = start + max_len
    if end > seq_len:
        end   = seq_len
        start = max(0, end - max_len)
    return sequence[start:end], start


def get_LLR_scores(sequence, model, tokenizer, device):
    seq_list  = list(sequence)
    tokenized = tokenizer(sequence, padding=False, truncation=True,
                          max_length=1024, return_tensors="pt")
    input_ids = tokenized["input_ids"].to(device)

    logits    = model(input_ids).logits
    log_probs = torch.log_softmax(logits, dim=-1)
    wt_logits = log_probs[:, 1:-1, :].squeeze(0)

    wt_logits_df = pd.DataFrame(
        wt_logits[:, 4:24].cpu().detach().numpy(),
        columns=AA_ORDER,
        index=[f"{aa} {i+1}" for i, aa in enumerate(seq_list)]
    ).T

    wt_norm = np.diag(wt_logits_df.loc[[c.split(" ")[0] for c in wt_logits_df.columns]])
    LLR     = wt_logits_df - wt_norm
    return LLR, wt_logits, seq_list


def fitness_matrix_to_rows(fitness, dom_seq, position_offset=0):
    rows = []
    for i, wt_aa in enumerate(dom_seq):
        if i >= len(fitness):
            break
        row = fitness[i]
        for j, mut_aa in enumerate(FITNESS_AA_ORDER):
            if j >= len(row):
                continue
            val = row[j]
            if val is None:
                continue
            try:
                fval = float(val)
            except (TypeError, ValueError):
                continue
            if np.isnan(fval):
                continue
            if mut_aa == wt_aa:
                continue
            rows.append({
                "position":           position_offset + i + 1,
                "wt_aa":              wt_aa,
                "mut_aa":             mut_aa,
                "normalized_fitness": fval,
            })
    return rows


def build_lora_model(lora_r, lora_alpha):
    """Rebuild the exact ESM2 + LoRA architecture used during training."""
    model       = EsmForMaskedLM.from_pretrained(MODEL_NAME)
    lora_config = LoraConfig(
        task_type      = "FEATURE_EXTRACTION",
        r              = lora_r,
        lora_alpha     = lora_alpha,
        target_modules = ["query", "key", "value", "output.dense"],
        lora_dropout   = LORA_DROPOUT,
    )
    model = get_peft_model(model, lora_config)
    return model


def recover_domains_from_metadata(metadata):
    """Pull per-domain {sequence, seq_offset, ...} out of a pfam's metadata.json.

    The training notebook writes a `domains` block per pfam that already
    contains the truncated input sequence and seq_offset used at training time,
    so we just normalize the fields here.
    """
    out = {}
    meta_domains = metadata.get("domains") or {}
    for domain_id, dinfo in meta_domains.items():
        if not isinstance(dinfo, dict) or "sequence" not in dinfo:
            continue
        out[domain_id] = {
            "uniprot_id":           dinfo.get("uniprot_id"),
            "sequence":             dinfo["sequence"],
            "seq_offset":           int(dinfo.get("seq_offset", 0)),
            "used_full_protein":    bool(dinfo.get("used_full_protein", False)),
            "dom_start_in_protein": int(dinfo.get("dom_start_in_protein", 0)),
        }
    return out


In [5]:
# ── Discover pfams from GCS ──────────────────────────────────────────────────
#
# The training notebook writes one directory per pfam under
# `GCS_OUTPUT_PREFIX{RUN_ID_PREFIX}/{pfam_id}/` (each containing a
# `metadata.json` + `best_model_lora.pth`). We list those blobs and recover
# the set of pfam ids that have a finetuned model available.

_run_prefix = f"{GCS_OUTPUT_PREFIX}{RUN_ID_PREFIX}/"

PFAM_IDS = sorted({
    blob.name[len(_run_prefix):].split("/", 1)[0]
    for blob in _gcs_client.list_blobs(GCS_BUCKET, prefix=_run_prefix)
    if blob.name.endswith("/metadata.json")
})

print(f"Found {len(PFAM_IDS)} pfams under gs://{GCS_BUCKET}/{_run_prefix}")
print(PFAM_IDS[:10], "..." if len(PFAM_IDS) > 10 else "")


Found 11 pfams under gs://domainome-data/ESM2/ft_per_pfam/global_finetune/
['PF00018', 'PF00046', 'PF00096', 'PF00240', 'PF00397', 'PF00595', 'PF00778', 'PF01846', 'PF02376', 'PF02817'] ...


In [ ]:
# ── Load tokenizer + base ESM2 (once, shared across pfams) ───────────────────

tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)

print("Loading base ESM2...")
base_model = EsmForMaskedLM.from_pretrained(MODEL_NAME).to(device)
base_model.eval()


In [ ]:
# ── Per-pfam loop: load LoRA, compute base/ft/Δ LLRs, save payload ───────────
#
# For every pfam discovered above we:
#   1. Download (or use cached) metadata.json + best_model_lora.pth
#   2. Rebuild the LoRA architecture and load the finetuned weights
#   3. Recover that pfam's per-domain {sequence, seq_offset, ...} from metadata
#   4. Compute base + finetuned LLRs for each of the pfam's domains
#   5. Pickle a single payload dict for the pfam and upload to GCS
#
# All per-pfam payloads are also kept in `pfam_payloads` for downstream cells.

pfam_payloads = {}

for pfam_id in tqdm(PFAM_IDS, desc="Pfams"):
    paths = pfam_paths(pfam_id)

    # 1. Metadata + LoRA weights
    if not os.path.exists(paths["metadata_local"]):
        gcs_download_to_file(paths["gcs_metadata"], paths["metadata_local"])
    if not os.path.exists(paths["lora_local"]):
        gcs_download_to_file(paths["gcs_lora"], paths["lora_local"])

    with open(paths["metadata_local"]) as f:
        metadata = json.load(f)

    best_cfg = metadata["best_config"]

    # 2. Rebuild LoRA + load finetuned weights for this pfam
    ft_model = build_lora_model(best_cfg["lora_r"], best_cfg["lora_alpha"])
    lora_state = torch.load(paths["lora_local"], map_location="cpu")
    ft_model.load_state_dict(lora_state, strict=False)
    ft_model.to(device)
    ft_model.eval()

    # 3. Recover this pfam's domains
    domains = recover_domains_from_metadata(metadata)
    if not domains:
        print(f"[{pfam_id}] no domains in metadata — skipping")
        del ft_model
        torch.cuda.empty_cache()
        continue

    # 4. Compute LLRs with both models
    llr_dict = {}
    with torch.no_grad():
        for domain_id, dinfo in domains.items():
            sequence   = dinfo["sequence"]
            seq_offset = dinfo["seq_offset"]

            base_LLR, _, _ = get_LLR_scores(sequence, base_model, tokenizer, device)
            ft_LLR,   _, _ = get_LLR_scores(sequence, ft_model,   tokenizer, device)
            delta_LLR      = ft_LLR - base_LLR

            llr_dict[domain_id] = {
                "uniprot_id":           dinfo.get("uniprot_id"),
                "sequence":             sequence,
                "seq_offset":           seq_offset,
                "used_full_protein":    dinfo.get("used_full_protein", False),
                "dom_start_in_protein": dinfo.get("dom_start_in_protein", 0),
                "aa_order":             AA_ORDER,
                "positions":            list(base_LLR.columns),
                "base_llr":             base_LLR.to_numpy().astype(np.float32),
                "ft_llr":               ft_LLR.to_numpy().astype(np.float32),
                "delta_llr":            delta_LLR.to_numpy().astype(np.float32),
            }

    # 5. Build the payload dict for this pfam, pickle locally + upload to GCS
    payload = {
        "run_id":      paths["run_id"],
        "pfam_id":     pfam_id,
        "model_name":  MODEL_NAME,
        "best_config": best_cfg,
        "aa_order":    AA_ORDER,
        "n_domains":   len(llr_dict),
        "domains":     llr_dict,
    }
    pfam_payloads[pfam_id] = payload

    with gzip.open(paths["llr_local"], "wb", compresslevel=6) as gz:
        pickle.dump(payload, gz, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = os.path.getsize(paths["llr_local"]) / 1e6
    gcs_upload_file(paths["gcs_llr"], paths["llr_local"], content_type="application/gzip")

    tqdm.write(
        f"[{pfam_id}] {len(llr_dict)} domains, {size_mb:.1f} MB → "
        f"gs://{GCS_BUCKET}/{paths['gcs_llr']}"
    )

    del ft_model
    torch.cuda.empty_cache()

print(f"\nDone — wrote payloads for {len(pfam_payloads)} pfams")


In [ ]:
# ── Quick sanity summary across all pfams ────────────────────────────────────

summary_rows = []
for pfam_id, payload in pfam_payloads.items():
    delta_means     = []
    delta_abs_means = []
    max_abs = 0.0
    for v in payload["domains"].values():
        delta_means.append(float(np.nanmean(v["delta_llr"])))
        delta_abs_means.append(float(np.nanmean(np.abs(v["delta_llr"]))))
        max_abs = max(max_abs, float(np.nanmax(np.abs(v["delta_llr"]))))
    summary_rows.append({
        "pfam_id":       pfam_id,
        "n_domains":     payload["n_domains"],
        "mean_delta":    float(np.mean(delta_means))     if delta_means     else float("nan"),
        "mean_abs_delta": float(np.mean(delta_abs_means)) if delta_abs_means else float("nan"),
        "max_abs_delta": max_abs,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("mean_abs_delta", ascending=False)
print(summary_df.head(20).to_string(index=False))
print(f"\nOverall mean ΔLLR:    {summary_df['mean_delta'].mean():+.4f}")
print(f"Overall mean |ΔLLR|:  {summary_df['mean_abs_delta'].mean():.4f}")


## Reload a pfam's comparison dict

```python
import gzip, io, pickle, pandas as pd
from google.cloud import storage

pfam_id = "PF00001"  # pick one

blob = storage.Client().bucket("<bucket>").blob(
    f"<prefix>/global_finetune/{pfam_id}/llr_comparison.pkl.gz"
)
with gzip.open(io.BytesIO(blob.download_as_bytes()), "rb") as gz:
    payload = pickle.load(gz)

entry    = payload["domains"]["<domain_id>"]
base_df  = pd.DataFrame(entry["base_llr"],  index=entry["aa_order"], columns=entry["positions"])
ft_df    = pd.DataFrame(entry["ft_llr"],    index=entry["aa_order"], columns=entry["positions"])
delta_df = pd.DataFrame(entry["delta_llr"], index=entry["aa_order"], columns=entry["positions"])
```


In [ ]:
# ── Reload per-pfam payloads (skip recompute) ────────────────────────────────
#
# For each pfam under PFAM_IDS, prefer the local pickle and fall back to
# downloading from GCS. Populates `pfam_payloads` so the Spearman + plotting
# cells below run without recomputing LLRs.

pfam_payloads = {}

for pfam_id in PFAM_IDS:
    paths = pfam_paths(pfam_id)
    if not os.path.exists(paths["llr_local"]):
        try:
            gcs_download_to_file(paths["gcs_llr"], paths["llr_local"])
        except Exception as exc:
            print(f"[{pfam_id}] download failed: {exc}")
            continue
    with gzip.open(paths["llr_local"], "rb") as gz:
        pfam_payloads[pfam_id] = pickle.load(gz)

print(f"Loaded payloads for {len(pfam_payloads)} pfams")
if pfam_payloads:
    sample_pfam = next(iter(pfam_payloads))
    sample_payload = pfam_payloads[sample_pfam]
    sample_id   = next(iter(sample_payload["domains"]))
    sample      = sample_payload["domains"][sample_id]
    print(f"Sample pfam={sample_pfam} domain={sample_id} "
          f"seq_len={len(sample['sequence'])} "
          f"base_llr={sample['base_llr'].shape} "
          f"ft_llr={sample['ft_llr'].shape}")


In [ ]:
# ── Spearman: fitness vs (base LLR, finetuned LLR) per domain × pfam ────────
#
# For every domain across every pfam in `pfam_payloads`, line up each measured
# (position, mut_aa) fitness value with the corresponding entry in the base and
# finetuned LLR matrices and compute Spearman ρ separately for base and ft.

from scipy.stats import spearmanr

try:
    fitness_dict
except NameError:
    print(f"Loading fitness dict from gs://{GCS_BUCKET}/{GCS_DOMAIN_DATA_BLOB}")
    fitness_dict = gcs_download_pickle_gz(_gcs_bucket.blob(GCS_DOMAIN_DATA_BLOB))
    print(f"Loaded {len(fitness_dict)} fitness entries")

aa_to_row = {aa: i for i, aa in enumerate(AA_ORDER)}

per_domain_rows = []
skipped_no_match = 0
skipped_too_few  = 0

for pfam_id, payload in tqdm(pfam_payloads.items(), desc="Pfams"):
    for domain_id, entry in payload["domains"].items():
        fd = fitness_dict.get(domain_id)
        if not isinstance(fd, dict):
            continue
        fitness = fd.get("fitness")
        dom_seq = fd.get("dom_seq")
        if fitness is None or dom_seq is None:
            continue

        sequence = entry["sequence"]
        base_llr = entry["base_llr"]
        ft_llr   = entry["ft_llr"]
        n_pos    = base_llr.shape[1]

        idx_in_seq = sequence.find(dom_seq)
        if idx_in_seq < 0:
            skipped_no_match += 1
            continue

        fits, base_vals, ft_vals = [], [], []
        for i, wt_aa in enumerate(dom_seq):
            col = idx_in_seq + i
            if col >= n_pos or i >= len(fitness):
                break
            row = fitness[i]
            for j, mut_aa in enumerate(FITNESS_AA_ORDER):
                if j >= len(row):
                    continue
                val = row[j]
                if val is None or mut_aa == wt_aa:
                    continue
                try:
                    fval = float(val)
                except (TypeError, ValueError):
                    continue
                if np.isnan(fval):
                    continue
                r = aa_to_row.get(mut_aa)
                if r is None:
                    continue
                fits.append(fval)
                base_vals.append(float(base_llr[r, col]))
                ft_vals.append(float(ft_llr[r, col]))

        if len(fits) < 10:
            skipped_too_few += 1
            continue

        fits      = np.asarray(fits)
        base_vals = np.asarray(base_vals)
        ft_vals   = np.asarray(ft_vals)

        rho_base, p_base = spearmanr(fits, base_vals)
        rho_ft,   p_ft   = spearmanr(fits, ft_vals)

        per_domain_rows.append({
            "pfam_id":    pfam_id,
            "domain_id":  domain_id,
            "uniprot_id": entry.get("uniprot_id"),
            "n":          int(len(fits)),
            "rho_base":   float(rho_base),
            "p_base":     float(p_base),
            "rho_ft":     float(rho_ft),
            "p_ft":       float(p_ft),
            "delta_rho":  float(rho_ft - rho_base),
        })

corr_df = pd.DataFrame(per_domain_rows).sort_values("delta_rho", ascending=False)

print(f"Computed Spearman for {len(corr_df)} domains "
      f"(skipped {skipped_no_match} no-match, {skipped_too_few} too-few)")
print(f"Mean ρ_base : {corr_df['rho_base'].mean():+.4f}   "
      f"median {corr_df['rho_base'].median():+.4f}")
print(f"Mean ρ_ft   : {corr_df['rho_ft'].mean():+.4f}   "
      f"median {corr_df['rho_ft'].median():+.4f}")
print(f"Mean Δρ     : {corr_df['delta_rho'].mean():+.4f}   "
      f"median {corr_df['delta_rho'].median():+.4f}")
print(f"Domains improved (Δρ > 0): "
      f"{(corr_df['delta_rho'] > 0).sum()} / {len(corr_df)} "
      f"({(corr_df['delta_rho'] > 0).mean()*100:.1f}%)")

corr_df.head(10)


In [ ]:
# ── Plot: per-domain Spearman, base vs finetuned ─────────────────────────────

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

lo = float(min(corr_df["rho_base"].min(), corr_df["rho_ft"].min())) - 0.05
hi = float(max(corr_df["rho_base"].max(), corr_df["rho_ft"].max())) + 0.05

axes[0].scatter(corr_df["rho_base"], corr_df["rho_ft"],
                alpha=0.5, s=12, edgecolor="none")
axes[0].plot([lo, hi], [lo, hi], "k--", linewidth=1)
axes[0].set_xlim(lo, hi); axes[0].set_ylim(lo, hi)
axes[0].set_xlabel("Spearman ρ  (base ESM2 LLR vs fitness)")
axes[0].set_ylabel("Spearman ρ  (finetuned LLR vs fitness)")
axes[0].set_title("Per-domain Spearman correlation")

axes[1].hist(corr_df["delta_rho"], bins=40, color="C2", edgecolor="black")
axes[1].axvline(0, color="k", linewidth=1)
axes[1].set_xlabel("Δρ  (finetuned − base)")
axes[1].set_ylabel("# domains")
axes[1].set_title(f"Δρ distribution  (mean = {corr_df['delta_rho'].mean():+.4f})")

plt.tight_layout()
plt.show()